# Week 3: Ask business questions with SQL

**Your job:** create a local SQLite table from your cleaned Week 2 CSV, then write **eight SQL queries**. Fill in the `query_1` through `query_8` cells yourself. Run cells from top to bottom with **Shift+Enter**. Save with **⌘S**.

SQL helps an analyst answer questions from a table. You will use Python to run the queries, so you do not need a separate database app. Your database file stays in `data/processed/` and will not be uploaded to GitHub.

**Before you start:** Make sure `data/processed/valid_sales.csv` exists. If it does not, rerun Section 7 of your Week 2 notebook.

## 1. Build the local SQLite database

Run the next cell. It finds the project folder, reads your cleaned CSV, and creates a table named `sales`. This may take a minute. A **table** is like a spreadsheet inside a database; each invoice line is one row.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

project_root = next(
    (folder for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (folder / "data" / "processed" / "valid_sales.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run Week 2 Section 7 to create data/processed/valid_sales.csv first.")

csv_path = project_root / "data" / "processed" / "valid_sales.csv"
db_path = project_root / "data" / "processed" / "retail.db"
sales = pd.read_csv(
    csv_path,
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
)
with sqlite3.connect(db_path) as conn:
    sales.to_sql("sales", conn, if_exists="replace", index=False)
print(f"Created the sales table with {len(sales):,} rows")
print(f"Database: {db_path}")

Created the sales table with 530,104 rows
Database: /Users/abepark/Documents/Codex/2026-10-28/referenced-chatgpt-conversation-this-is-an/ecommerce-sales-analysis/data/processed/retail.db


## 2. Run your first query

Run the example. `SELECT` chooses what to show, `COUNT(*)` counts rows, `AS` names the result, and `FROM sales` identifies the table. The helper will show your results as a small table.

In [4]:
def run_query(query):
    # A blank exercise cell shows a prompt instead of an error.
    sql_lines = [line for line in query.splitlines()
                 if line.strip() and not line.lstrip().startswith("--")]
    if not sql_lines:
        print("Write your SQL below the comment, then run this cell again.")
        return
    with sqlite3.connect(db_path) as conn:
        result = pd.read_sql_query(query, conn)
    display(result)
    return result

run_query("SELECT COUNT(*) AS line_count FROM sales;")

,line_count
0,530104


,line_count
0,530104


### SQL building blocks

- `SELECT` chooses columns or calculations.
- `FROM sales` identifies the table.
- `WHERE` filters rows **before** grouping.
- `GROUP BY` makes one group per category or month.
- `SUM(Revenue)` adds line revenue within each group.
- `COUNT(DISTINCT InvoiceNo)` counts orders rather than invoice lines.
- `ORDER BY ... DESC` puts the largest result first.
- `LIMIT 10` shows ten rows.

A question to keep in mind: **What does one row of your result represent?** A country? A month? A customer?

## 3. Your eight queries

Write SQL in each code cell below. Keep the `--` comment as a reminder or delete it. The helper will run your SQL when you add a statement. If a query errors, read the last line of the error: it often names a misspelled column or missing comma.

### Query 1 — Overall sales KPIs

Return **one row** with total positive-sales revenue, number of distinct invoices, and number of distinct identified customers. Name the columns `total_revenue`, `orders`, and `identified_customers`.

**Hints:** `SUM(Revenue)`, `COUNT(DISTINCT InvoiceNo)`, and `COUNT(DISTINCT CustomerID)`. `COUNT` ignores missing customer IDs.

In [21]:
query_1 = """
SELECT
SUM(Revenue) AS total_revenue,
Count(DISTINCT InvoiceNo) AS orders,
Count(DISTINCT CustomerID) AS identified_customers
FROM sales
"""
run_query(query_1)

,total_revenue,orders,identified_customers
0,1.066668e+07,19960,4338


,total_revenue,orders,identified_customers
0,1.066668e+07,19960,4338


### Query 2 — Top countries

Return the **five countries** with the most positive-sales revenue. Show `Country` and `total_revenue`.

**Hints:** Group by `Country`, sort the sum descending, and use `LIMIT 5`. Which country dominates?

In [23]:
query_2 = """
SELECT
    Country,
    SUM(Revenue) AS total_revenue
FROM sales

GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 5

"""
run_query(query_2)

,Country,total_revenue
0,United Kingdom,9025222.084
1,Netherlands,285446.340
2,EIRE,283453.960
3,Germany,228867.140
4,France,209715.110


,Country,total_revenue
0,United Kingdom,9025222.084
1,Netherlands,285446.340
2,EIRE,283453.960
3,Germany,228867.140
4,France,209715.110


### Query 3 — Monthly revenue

Return one row per `YearMonth`, in date order, with the sum of `Revenue`. Name the total `total_revenue`.

**Hints:** `GROUP BY YearMonth` and `ORDER BY YearMonth`. The values are formatted `YYYY-MM`, so alphabetical order is also date order. December 2011 is an incomplete month.

In [25]:
query_3 = """
SELECT 
YearMonth,
Sum(Revenue) as total_revenue
FROM sales

GROUP BY YearMonth
ORDER BY YearMonth

"""
run_query(query_3)

,YearMonth,total_revenue
0,2010-12,823746.140
1,2011-01,691364.560
2,2011-02,523631.890
3,2011-03,717639.360
4,2011-04,537808.621
5,2011-05,770536.020
6,2011-06,761739.900
7,2011-07,719221.191
8,2011-08,759138.380
9,2011-09,1058590.172


,YearMonth,total_revenue
0,2010-12,823746.140
1,2011-01,691364.560
2,2011-02,523631.890
3,2011-03,717639.360
4,2011-04,537808.621
5,2011-05,770536.020
6,2011-06,761739.900
7,2011-07,719221.191
8,2011-08,759138.380
9,2011-09,1058590.172


### Query 4 — Monthly order count and average order value

For each month, show `YearMonth`, distinct order count, and **average order value** (AOV). AOV is total revenue divided by distinct orders, not by invoice lines.

**Hints:** Reuse Query 3, add `COUNT(DISTINCT InvoiceNo)`, then divide `SUM(Revenue)` by that count. `ROUND(..., 2)` displays two decimals.

In [50]:
query_4 = """
SELECT
YearMonth,
COUNT(DISTINCT InvoiceNo) AS order_count,
SUM(Revenue) AS total_revenue,
ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS average_order_value
FROM sales

GROUP BY YearMonth
ORDER BY YearMonth

"""
run_query(query_4)

,YearMonth,order_count,total_revenue,average_order_value
0,2010-12,1559,823746.140,528.38
1,2011-01,1086,691364.560,636.62
2,2011-02,1100,523631.890,476.03
3,2011-03,1454,717639.360,493.56
4,2011-04,1246,537808.621,431.63
5,2011-05,1681,770536.020,458.38
6,2011-06,1533,761739.900,496.89
7,2011-07,1475,719221.191,487.61
8,2011-08,1361,759138.380,557.78
9,2011-09,1837,1058590.172,576.26


,YearMonth,order_count,total_revenue,average_order_value
0,2010-12,1559,823746.140,528.38
1,2011-01,1086,691364.560,636.62
2,2011-02,1100,523631.890,476.03
3,2011-03,1454,717639.360,493.56
4,2011-04,1246,537808.621,431.63
5,2011-05,1681,770536.020,458.38
6,2011-06,1533,761739.900,496.89
7,2011-07,1475,719221.191,487.61
8,2011-08,1361,759138.380,557.78
9,2011-09,1837,1058590.172,576.26


### Query 5 — Top products

Find the ten product codes with the highest revenue. Show `StockCode` and `total_revenue`.

**Hint:** Group by `StockCode`. Descriptions may vary for the same code, so use the code as the product key. Inspect any unusual codes before calling them best-selling products.

In [44]:
query_5 = """
SELECT
StockCode,
Sum(Revenue) AS total_revenue
FROM sales

GROUP BY StockCode
ORDER BY total_revenue

"""
run_query(query_5)

,StockCode,total_revenue
0,PADS,0.003
1,84227,0.420
2,51014c,0.830
3,21268,0.840
4,90084,0.850
...,...,...
3917,47566,99504.330
3918,85123A,104518.800
3919,23843,168469.600
3920,22423,174484.740


,StockCode,total_revenue
0,PADS,0.003
1,84227,0.420
2,51014c,0.830
3,21268,0.840
4,90084,0.850
...,...,...
3917,47566,99504.330
3918,85123A,104518.800
3919,23843,168469.600
3920,22423,174484.740


### Query 6 — Missing customer IDs

Return **one row** with the count of positive-sales lines whose `CustomerID` is missing and their total revenue.

**Hint:** `WHERE CustomerID IS NULL`. Think about why these rows stay in overall sales but cannot be assigned to an identified customer.

In [51]:
query_6 = """
SELECT
CustomerID
FROM sales

WHERE CustomerID IS NULL


"""
run_query(query_6)

,CustomerID
0,None
1,None
2,None
3,None
4,None
...,...
132215,None
132216,None
132217,None
132218,None


,CustomerID
0,None
1,None
2,None
3,None
4,None
...,...
132215,None
132216,None
132217,None
132218,None


### Query 7 — Repeat customers

Find identified customers with **more than one distinct invoice**. Show `CustomerID` and the number of distinct invoices, largest first. Show the first ten.

**Hints:** Filter out `NULL` IDs with `WHERE`, group by `CustomerID`, then use `HAVING COUNT(DISTINCT InvoiceNo) > 1`. `HAVING` filters groups **after** grouping.

In [ ]:
query_7 = """
SELECT
CustomerID,
COUNT(DISTINCT InvoiceNo) AS invoices
FROM sales

WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
HAVING invoices > 1

"""
run_query(query_7)

,CustomerID,invoices
0,12748,209
1,14911,201
2,17841,124
3,13089,97
4,14606,93
...,...,...
2840,12375,2
2841,12371,2
2842,12365,2
2843,12363,2


,CustomerID,invoices
0,12748,209
1,14911,201
2,17841,124
3,13089,97
4,14606,93
...,...,...
2840,12375,2
2841,12371,2
2842,12365,2
2843,12363,2


### Query 8 — Your own business question

Choose one question that would help the retailer. Examples: Which months had the most orders? Which countries had the highest average order value? Which products sold the most units? Write the question in the Markdown cell below, then write your SQL.

**Hint:** State what one row of the result should represent before writing `SELECT`.

**My business question:** Which month had the most orders?

**What one result row represents:** Orders of a month

In [71]:
query_8 = """
SELECT
YearMonth,
COUNT(DISTINCT InvoiceNo) AS order_count
FROM sales

GROUP BY YearMonth
ORDER BY order_count DESC

"""
run_query(query_8)

,YearMonth,order_count
0,2011-11,2769
1,2011-10,2040
2,2011-09,1837
3,2011-05,1681
4,2010-12,1559
5,2011-06,1533
6,2011-07,1475
7,2011-03,1454
8,2011-08,1361
9,2011-04,1246


,YearMonth,order_count
0,2011-11,2769
1,2011-10,2040
2,2011-09,1837
3,2011-05,1681
4,2010-12,1559
5,2011-06,1533
6,2011-07,1475
7,2011-03,1454
8,2011-08,1361
9,2011-04,1246


## 4. Save your queries and explain the results

Copy each working SQL statement into `sql/analysis_queries.sql` under its matching question. Keep the database in `data/processed/`; it is generated locally and ignored by Git.

Write **three findings** below. Each finding should state a number, explain what it means, and name one limitation or follow-up. For example, be careful with December 2011 because the data ends on December 9.

**Finding 1:** ___

**Finding 2:** ___

**Finding 3:** ___

**One limitation of this analysis:** ___